In [ ]:
!nvidia-smi
!python --version
import torch; print(torch.__version__, torch.cuda.is_available())

Sat Nov 14 15:24:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Downloading the COCO

In [ ]:
!mkdir -p /content/data && cd /content/data && wget -q http://images.cocodataset.org/zips/val2017.zip http://images.cocodataset.org/annotations/annotations_trainval2017.zip && unzip -q val2017.zip && unzip -q annotations_trainval2017.zip


In [ ]:
!git clone https://github.com/mmunir127/rapidnet-official.git
!wget -q -O RapidNet_M_Detection.pth https://huggingface.co/SLDGroup/RapidNet/resolve/main/Detection/RapidNet_M_Detection.pth

Cloning into 'rapidnet-official'...
remote: Enumerating objects: 110, done.
remote: Counting objects: 100% (110/110), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 110 (delta 24), reused 95 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (110/110), 86.84 KiB | 6.20 MiB/s, done.
Resolving deltas: 100% (24/24), done.


### Create the Python 3.9 environment and install torch 1.12.1

In [ ]:
!pip install -q uv
!uv venv /content/py39 --python 3.9
!uv pip install --python /content/py39/bin/python torch==1.12.1+cu116 torchvision==0.13.1+cu116 --extra-index-url https://download.pytorch.org/whl/cu116

Using CPython 3.9.25
Creating virtual environment at: py39
Activate with: source py39/bin/activate
Using Python 3.9.25 environment at: py39
Resolved 10 packages in 1.23s
Prepared 10 packages in 47.31s
Installed 10 packages in 188ms
 + certifi==2022.12.7
 + charset-normalizer==2.1.1
 + idna==3.4
 + numpy==2.0.2
 + pillow==11.3.0
 + requests==2.28.1
 + torch==1.12.1+cu116
 + torchvision==0.13.1+cu116
 + typing-extensions==4.16.0
 + urllib3==1.26.13


In [ ]:
!uv pip install --python /content/py39/bin/python "numpy<1.24" timm submitit pycocotools opencv-python-headless
!uv pip install --python /content/py39/bin/python --no-build mmcv-full==1.7.1 -f https://download.openmmlab.com/mmcv/dist/cu116/torch1.12.0/index.html
!uv pip install --python /content/py39/bin/python mmdet==2.28

Using Python 3.9.25 environment at: py39
Resolved 36 packages in 579ms
Prepared 27 packages in 3.90s
Uninstalled 1 package in 13ms
Installed 27 packages in 39ms
 + annotated-doc==0.0.5
 + anyio==4.12.1
 + click==8.1.8
 + cloudpickle==3.1.2
 + exceptiongroup==1.3.1
 + filelock==3.19.1
 + fsspec==2025.10.0
 + h11==0.16.0
 + hf-xet==1.8.0
 + httpcore==1.0.9
 + httpx==0.28.1
 + huggingface-hub==1.8.0
 + markdown-it-py==3.0.0
 + mdurl==0.1.2
 - numpy==2.0.2
 + numpy==1.23.5
 + opencv-python-headless==4.11.0.86
 + packaging==26.3
 + pycocotools==2.0.11
 + pygments==2.21.0
 + pyyaml==6.0.3
 + rich==15.0.0
 + safetensors==0.7.0
 + shellingham==1.5.4
 + submitit==1.5.4
 + timm==1.0.30
 + tqdm==4.70.1
 + typer==0.23.2
Using Python 3.9.25 environment at: py39
Resolved 10 packages in 647ms
Prepared 7 packages in 11.28s
Uninstalled 1 package in 13ms
Installed 7 packages in 27ms
 + addict==2.4.0
 + mmcv-full==1.7.1
 - numpy==1.23.5
 + numpy==2.0.2
 + opencv-python==5.0.0.93
 + platformdirs==4.4.0
 +

In [ ]:
!uv pip install --python /content/py39/bin/python "setuptools<70" wheel

Using Python 3.9.25 environment at: py39
Resolved 3 packages in 143ms
Prepared 2 packages in 53ms
Installed 2 packages in 6ms
 + setuptools==69.5.1
 + wheel==0.48.0


In [ ]:
!/content/py39/bin/python -c "import torch, mmcv, mmdet; print(torch.__version__, torch.cuda.is_available(), mmcv.__version__, mmdet.__version__)"

/content/py39/lib/python3.9/site-packages/mmcv/__init__.py:20: UserWarning: On January 1, 2023, MMCV will release v2.0.0, in which it will remove components related to the training process and add a data transformation module. In addition, it will rename the package names mmcv to mmcv-lite and mmcv-full to mmcv. See https://github.com/open-mmlab/mmcv/blob/master/docs/en/compatibility.md for more details.
  warnings.warn(
1.12.1+cu116 True 1.7.1 2.28.0


In [ ]:
!ls /content/rapidnet-official/detection
!ls /content/rapidnet-official/detection/configs
!grep -n "data_root" /content/rapidnet-official/detection/configs/*.py | head
!ls /content/data /content/*.pth


checkpoint.py  main.py	    mmdet_custom	  test.py
configs        mmcv_custom  rapidnet_backbone.py
_base_				     mask_rcnn_rapidnet_m_fpn_1x_coco.py
mask_rcnn_rapidnet_b_fpn_1x_coco.py
/content/RapidNet_M_Detection.pth

/content/data:
annotations  annotations_trainval2017.zip  val2017  val2017.zip


In [ ]:
%cd /content/rapidnet-official/detection
!mkdir -p data/coco
!ln -sfn /content/data/val2017 data/coco/val2017
!ln -sfn /content/data/annotations data/coco/annotations
!grep -rn "data_root" configs/_base_ | head
!/content/py39/bin/python test.py --help | head -40
%env MPLBACKEND=Agg
%cd /content/rapidnet-official/detection
!grep -rn "data_root" configs/_base_ | head
!/content/py39/bin/python test.py --help | head -40

/content/rapidnet-official/detection
configs/_base_/datasets/coco_instance_semantic.py:3:data_root = 'data/coco/'
configs/_base_/datasets/coco_instance_semantic.py:40:        ann_file=data_root + 'annotations/instances_train2017.json',
configs/_base_/datasets/coco_instance_semantic.py:41:        img_prefix=data_root + 'train2017/',
configs/_base_/datasets/coco_instance_semantic.py:42:        seg_prefix=data_root + 'stuffthingmaps/train2017/',
configs/_base_/datasets/coco_instance_semantic.py:46:        ann_file=data_root + 'annotations/instances_val2017.json',
configs/_base_/datasets/coco_instance_semantic.py:47:        img_prefix=data_root + 'val2017/',
configs/_base_/datasets/coco_instance_semantic.py:51:        ann_file=data_root + 'annotations/instances_val2017.json',
configs/_base_/datasets/coco_instance_semantic.py:52:        img_prefix=data_root + 'val2017/',
configs/_base_/datasets/coco_instance.py:3:data_root = '../../../Datasets/coco/'
configs/_base_/datasets/coco_instance.py

Numpy

In [ ]:
!uv pip install --python /content/py39/bin/python "numpy==1.23.5"

Using Python 3.9.25 environment at: /content/py39
Resolved 1 package in 4ms
Uninstalled 1 package in 16ms
Installed 1 package in 14ms
 - numpy==2.0.2
 + numpy==1.23.5


In [ ]:
!wget -q -O /content/rapidnet-official/RapidNet_M.pth https://huggingface.co/SLDGroup/RapidNet/resolve/main/RapidNet_M.pth
!ls -la /content/rapidnet-official/RapidNet_M.pth

-rw-r--r-- 1 root root 139591669 Nov 14 15:27 /content/rapidnet-official/RapidNet_M.pth


In [ ]:
%env MPLBACKEND=Agg
%cd /content/rapidnet-official/detection
!/content/py39/bin/python test.py configs/mask_rcnn_rapidnet_m_fpn_1x_coco.py --checkpoint /content/RapidNet_M_Detection.pth --eval bbox segm --launcher none --work-dir /content/eval_m --cfg-options data.test.ann_file=/content/data/annotations/instances_val2017.json data.test.img_prefix=/content/data/val2017/

env: MPLBACKEND=Agg
/content/rapidnet-official/detection
/content/py39/lib/python3.9/site-packages/mmcv/__init__.py:20: UserWarning: On January 1, 2023, MMCV will release v2.0.0, in which it will remove components related to the training process and add a data transformation module. In addition, it will rename the package names mmcv to mmcv-lite and mmcv-full to mmcv. See https://github.com/open-mmlab/mmcv/blob/master/docs/en/compatibility.md for more details.
  warnings.warn(
/content/py39/lib/python3.9/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/content/py39/lib/python3.9/site-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via

### RapidNetB

In [ ]:
!wget -q -O /content/RapidNet_B_Detection.pth https://huggingface.co/SLDGroup/RapidNet/resolve/main/Detection/RapidNet_B_Detection.pth
!wget -q -O /content/rapidnet-official/RapidNet_B.pth https://huggingface.co/SLDGroup/RapidNet/resolve/main/RapidNet_B.pth

In [ ]:
%env MPLBACKEND=Agg
%cd /content/rapidnet-official/detection
!/content/py39/bin/python test.py configs/mask_rcnn_rapidnet_b_fpn_1x_coco.py --checkpoint /content/RapidNet_B_Detection.pth --eval bbox segm --launcher none --work-dir /content/eval_b --cfg-options data.test.ann_file=/content/data/annotations/instances_val2017.json data.test.img_prefix=/content/data/val2017/

env: MPLBACKEND=Agg
/content/rapidnet-official/detection
/content/py39/lib/python3.9/site-packages/mmcv/__init__.py:20: UserWarning: On January 1, 2023, MMCV will release v2.0.0, in which it will remove components related to the training process and add a data transformation module. In addition, it will rename the package names mmcv to mmcv-lite and mmcv-full to mmcv. See https://github.com/open-mmlab/mmcv/blob/master/docs/en/compatibility.md for more details.
  warnings.warn(
/content/py39/lib/python3.9/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/content/py39/lib/python3.9/site-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via